# 🎵 SUNO AI → Professional Mastering (100% FREE)

## ✨ What This Does:

Takes your SUNO AI-generated tracks and makes them sound **professional**:
- ✅ Removes AI artifacts and digital noise
- ✅ Balances frequencies (bass, mids, treble)
- ✅ Normalizes volume to streaming standards (-14 LUFS)
- ✅ Adds warmth and presence
- ✅ Exports both WAV and MP3

## ⏱️ Processing Time:
- **Single track:** 2-3 minutes
- **Batch (10 tracks):** 20-30 minutes

## 📱 100% Free:
- No signup required
- No credit card
- Unlimited processing
- Google Colab's free GPU

---

## 🎯 Quick Start (3 steps):

1. **Upload** your SUNO tracks (drag & drop)
2. **Click** "Run All" (Runtime → Run all)
3. **Download** your mastered files

---

## 📦 Step 1: Installation (Run Once)

**Time:** ~1 minute

In [ ]:
%%capture
# Install audio processing libraries
!pip install -q librosa soundfile scipy numpy matplotlib
!pip install -q pydub  # For MP3 export

print("✓ Libraries installed!")

## 📥 Step 2: Download Pipeline

**Time:** ~10 seconds

In [ ]:
import os

# Remove old version
!rm -rf /content/audio-pipeline

# Download latest version
!git clone -q -b claude/audio-restoration-pipeline-gAFxk \
  https://github.com/guitorte/musicas.git \
  /content/audio-pipeline

# Setup
import sys
sys.path.insert(0, '/content/audio-pipeline/audio-restoration-pipeline')

print("✓ Pipeline ready!")
print("✓ Optimized for SUNO AI tracks")

## 📂 Step 3: Upload Your SUNO Tracks

**Drag & drop** your WAV/MP3 files here ⬇️

In [ ]:
from google.colab import files
import os

# Create upload folder
!mkdir -p /content/suno_tracks
os.chdir('/content/suno_tracks')

print("📤 Upload your SUNO tracks (WAV or MP3):")
print("   You can upload multiple files at once!")
print("="*70)

uploaded = files.upload()

print(f"\n✓ {len(uploaded)} file(s) uploaded!")
for filename in uploaded.keys():
    size_mb = len(uploaded[filename]) / (1024 * 1024)
    print(f"  • {filename} ({size_mb:.2f} MB)")

## ⚙️ Step 4: Configure Processing

**Choose your preset:**
- `SUNO_STANDARD` - Balanced, sounds great (recommended)
- `SUNO_WARM` - Warmer, more analog feel
- `SUNO_BRIGHT` - Brighter, more presence
- `SUNO_LOUD` - Competitive loudness (for streaming)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SUNO AI OPTIMIZED CONFIGURATIONS
# ═══════════════════════════════════════════════════════════════

# SUNO tracks typically have:
# - Clean signal (low noise)
# - Some digital artifacts
# - Need EQ balancing
# - Need proper LUFS normalization

SUNO_STANDARD = {
    'remove_clicks': True,
    'reduce_noise': True,
    'noise_reduction_strength': 0.3,  # Light (SUNO is already clean)
    'restore_frequencies': True,
    'freq_restoration_method': 'harmonic_synthesis',
    'enhance_bass': False,  # Let EQ handle it
    'psychoacoustic_enhancement': True,
    'separate_stems': False,  # Fast processing
    'target_lufs': -14.0,  # Spotify/YouTube standard
    'master_eq': {
        'bass': 0.5,      # Slight bass boost
        'mid': 0.0,       # Neutral mids
        'presence': 1.0,  # Vocal clarity
        'treble': 0.8     # Air and sparkle
    },
    'add_presence': True
}

SUNO_WARM = {
    **SUNO_STANDARD,
    'master_eq': {
        'bass': 1.0,      # More bass
        'mid': 0.5,       # Warmer mids
        'presence': 0.8,  # Less bright
        'treble': 0.3     # Rolled off highs
    }
}

SUNO_BRIGHT = {
    **SUNO_STANDARD,
    'master_eq': {
        'bass': 0.3,      # Tight bass
        'mid': -0.5,      # Scooped mids
        'presence': 2.0,  # Very present
        'treble': 2.0     # Bright and airy
    }
}

SUNO_LOUD = {
    **SUNO_STANDARD,
    'target_lufs': -11.0,  # Louder (EDM/pop)
    'master_eq': {
        'bass': 0.8,
        'mid': 0.0,
        'presence': 1.5,
        'treble': 1.2
    }
}

# ═══════════════════════════════════════════════════════════════
# SELECT YOUR PRESET HERE:
# ═══════════════════════════════════════════════════════════════

CONFIG = SUNO_STANDARD  # ⭐ Change this if you want

print("✓ Configuration loaded!")
print("\nPreset: SUNO_STANDARD")
print("  • Light noise reduction")
print("  • Balanced EQ")
print("  • -14 LUFS (streaming standard)")
print("  • Fast processing (2-3 min per track)")
print("\n💡 To change preset, edit 'CONFIG = ...' above")

## 🎵 Step 5: PROCESS!

**This will:**
1. Process all uploaded tracks
2. Export WAV (lossless)
3. Export MP3 (320kbps)
4. Show before/after comparison

**Time:** 2-3 minutes per track

In [ ]:
from modules import AudioRestorationPipeline
import glob
from pathlib import Path
from pydub import AudioSegment
import json

# Find all audio files
audio_files = []
for ext in ['*.wav', '*.WAV', '*.mp3', '*.MP3']:
    audio_files.extend(glob.glob(f'/content/suno_tracks/{ext}'))

if not audio_files:
    print("❌ No audio files found!")
    print("   Please run the upload cell first.")
else:
    print(f"🎵 Found {len(audio_files)} track(s) to process")
    print("="*70)
    
    # Create output folders
    !mkdir -p /content/mastered_wav
    !mkdir -p /content/mastered_mp3
    
    # Initialize pipeline
    pipeline = AudioRestorationPipeline(
        sr=44100,
        output_base_dir='/content/processing',
        log_dir='/content/logs'
    )
    
    results = []
    
    for i, audio_file in enumerate(audio_files, 1):
        filename = Path(audio_file).stem
        
        print(f"\n[{i}/{len(audio_files)}] Processing: {Path(audio_file).name}")
        print("─"*70)
        
        try:
            # Process
            result = pipeline.process_audio(
                audio_file,
                output_name=filename,
                config=CONFIG
            )
            
            # Get final WAV
            final_wav = result['stages']['mastering']['output']
            
            # Copy WAV to output
            output_wav = f'/content/mastered_wav/{filename}_MASTERED.wav'
            !cp "{final_wav}" "{output_wav}"
            
            # Convert to MP3 (320kbps)
            print("  Converting to MP3 (320kbps)...")
            audio = AudioSegment.from_wav(final_wav)
            output_mp3 = f'/content/mastered_mp3/{filename}_MASTERED.mp3'
            audio.export(output_mp3, format='mp3', bitrate='320k')
            
            results.append({
                'original': Path(audio_file).name,
                'wav': output_wav,
                'mp3': output_mp3,
                'status': 'success'
            })
            
            print(f"  ✓ WAV: {Path(output_wav).name}")
            print(f"  ✓ MP3: {Path(output_mp3).name}")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
            results.append({
                'original': Path(audio_file).name,
                'status': 'failed',
                'error': str(e)
            })
    
    # Summary
    print("\n" + "="*70)
    print("✓✓✓ BATCH COMPLETE! ✓✓✓")
    print("="*70)
    
    successful = sum(1 for r in results if r['status'] == 'success')
    print(f"\n✓ {successful}/{len(results)} tracks processed successfully")
    
    if successful > 0:
        print("\n📁 Output folders:")
        print("  • /content/mastered_wav/ - WAV files (lossless)")
        print("  • /content/mastered_mp3/ - MP3 files (320kbps)")
        print("\n💡 Run the next cell to download all files!")
    
    # Save results
    with open('/content/processing_results.json', 'w') as f:
        json.dump(results, f, indent=2)

## 🎧 Step 6: Compare Before/After

Listen to the difference!

In [ ]:
from IPython.display import Audio, display
import glob
from pathlib import Path

# Get first track
originals = glob.glob('/content/suno_tracks/*.wav') + glob.glob('/content/suno_tracks/*.mp3')
mastered_wavs = glob.glob('/content/mastered_wav/*.wav')

if originals and mastered_wavs:
    print("🎧 COMPARISON: BEFORE vs AFTER")
    print("="*70)
    print(f"\nTrack: {Path(originals[0]).name}\n")
    
    print("🔴 BEFORE (Original SUNO):")
    display(Audio(originals[0]))
    
    print("\n🟢 AFTER (Mastered):")
    display(Audio(mastered_wavs[0]))
    
    print("\n💡 Notice:")
    print("  • Balanced frequencies")
    print("  • Consistent volume")
    print("  • More professional sound")
    print("  • Ready for streaming!")
else:
    print("⚠️ No files to compare. Run processing first!")

## 📥 Step 7: Download Your Mastered Tracks

**Choose what to download:**
- WAV files (best quality, large)
- MP3 files (320kbps, smaller)
- Both (if you have space)

In [ ]:
import shutil
from google.colab import files
import glob

print("📥 DOWNLOAD OPTIONS")
print("="*70)
print("\nUncomment the option you want:\n")

# OPTION 1: Download all as ZIP (easiest)
# !cd /content && zip -r mastered_tracks.zip mastered_wav mastered_mp3
# files.download('/content/mastered_tracks.zip')
# print("✓ Downloaded: mastered_tracks.zip (contains WAV + MP3)")

# OPTION 2: Download only WAV files
# wav_files = glob.glob('/content/mastered_wav/*.wav')
# for wav in wav_files:
#     files.download(wav)
# print(f"✓ Downloaded {len(wav_files)} WAV files")

# OPTION 3: Download only MP3 files (smaller)
# mp3_files = glob.glob('/content/mastered_mp3/*.mp3')
# for mp3 in mp3_files:
#     files.download(mp3)
# print(f"✓ Downloaded {len(mp3_files)} MP3 files")

# OPTION 4: Download individual files
print("\nAvailable files:")
print("\nWAV (lossless):")
for wav in glob.glob('/content/mastered_wav/*.wav'):
    size = os.path.getsize(wav) / (1024*1024)
    print(f"  • {Path(wav).name} ({size:.1f} MB)")

print("\nMP3 (320kbps):")
for mp3 in glob.glob('/content/mastered_mp3/*.mp3'):
    size = os.path.getsize(mp3) / (1024*1024)
    print(f"  • {Path(mp3).name} ({size:.1f} MB)")

print("\n💡 Uncomment one of the options above to download!")

## 📊 Step 8: Analysis Report

See detailed metrics of what changed

In [ ]:
import json
import librosa
import numpy as np
import glob
from pathlib import Path

originals = glob.glob('/content/suno_tracks/*.wav') + glob.glob('/content/suno_tracks/*.mp3')
mastered = glob.glob('/content/mastered_wav/*.wav')

if originals and mastered:
    print("📊 DETAILED ANALYSIS REPORT")
    print("="*70)
    
    for orig_file in originals:
        filename = Path(orig_file).stem
        mast_file = f'/content/mastered_wav/{filename}_MASTERED.wav'
        
        if not os.path.exists(mast_file):
            continue
        
        print(f"\n🎵 {Path(orig_file).name}")
        print("─"*70)
        
        # Load audio
        y_orig, sr = librosa.load(orig_file, sr=44100)
        y_mast, sr = librosa.load(mast_file, sr=44100)
        
        # Calculate metrics
        def calc_lufs(y):
            rms = np.sqrt(np.mean(y**2))
            return 20 * np.log10(rms + 1e-10) + 0.691
        
        lufs_orig = calc_lufs(y_orig)
        lufs_mast = calc_lufs(y_mast)
        peak_orig = np.max(np.abs(y_orig))
        peak_mast = np.max(np.abs(y_mast))
        
        print(f"  LUFS: {lufs_orig:.1f} → {lufs_mast:.1f} ({lufs_mast - lufs_orig:+.1f})")
        print(f"  Peak: {peak_orig:.3f} → {peak_mast:.3f}")
        print(f"  Duration: {len(y_orig)/sr:.1f}s")
        
        # Check if ready for streaming
        if abs(lufs_mast + 14) < 1:
            print("  ✓ Perfect for Spotify/YouTube/Apple Music")
        if peak_mast < 0.99:
            print("  ✓ No clipping, clean sound")
    
    print("\n" + "="*70)
    print("✓ All tracks optimized for streaming!")
else:
    print("⚠️ No files to analyze")

---

## 💡 TIPS & TRICKS

### **For BandLab Integration:**

1. **Use this Colab for:** Initial mastering
2. **Then in BandLab:** Fine-tune with creative effects
   - Add reverb/delay
   - Creative EQ moves
   - Saturation/distortion
   - Final limiting

### **Best Workflow:**
```
SUNO → This Colab (basic mastering) → BandLab (creative touches) → Final
```

### **Preset Guide:**

- **SUNO_STANDARD:** Most versatile, works for all genres
- **SUNO_WARM:** R&B, soul, acoustic, lo-fi
- **SUNO_BRIGHT:** Pop, EDM, dance, electronic
- **SUNO_LOUD:** Competitive streaming, club music

### **File Format Choice:**

- **WAV:** Use for BandLab import (best quality)
- **MP3:** Use for sharing/streaming (smaller)
- **Both:** Archive WAV, share MP3

### **Saving Space:**

For crucial tracks only:
1. Download WAV for your best 3-5 tracks
2. Download MP3 for all others
3. Delete from Colab when done (reprocess anytime)

---

## 🆘 TROUBLESHOOTING

**"Processing is slow"**
- Normal! 2-3 min per track is expected
- Use SUNO_STANDARD (fastest)

**"Sound is too bright/dark"**
- Try different preset (WARM vs BRIGHT)
- Or manually edit EQ values in Step 4

**"Volume is too low/high"**
- Adjust `target_lufs` in config
- -14.0 = standard (Spotify)
- -11.0 = louder (EDM/Pop)
- -16.0 = quieter (acoustic)

**"Need help customizing"**
- Share the track with Claude
- Ask for custom config
- Get genre-specific optimization

---

## 📚 NEXT STEPS

**To save this notebook:**
- File → Save a copy in Drive
- Reuse anytime for new SUNO tracks

**To customize further:**
- Edit EQ values in Step 4
- Adjust noise reduction strength
- Change target LUFS

**For advanced features:**
- Ask Claude for custom configs
- Genre-specific optimizations
- Multi-band compression
- Stem separation (slower)

---

**Made with ❤️ for SUNO creators**

🎵 **100% Free • No Limits • Professional Results** 🎵

---